# NB5b-LOGO — Leave-One-Generator-Out on the real AraBERT [CLS] embeddings

## What this notebook does

The frozen probe in NB5b scored 99.5% macro-F1, and a diagnostic TF-IDF classifier scored the same
number. That tight match suggested both classifiers were leaning on the same surface-lexical signal.
A quick TF-IDF Leave-One-Generator-Out then held at 99.0%, indicating the signal generalizes rather
than merely memorizing a per-generator fingerprint.

But **that check ran on TF-IDF, not on AraBERT itself**, so it does not directly prove that the
frozen [CLS] representation generalizes. This notebook fills that gap. It reads the AraBERT [CLS]
embeddings I already cached in NB5b and runs LOGO on **those exact vectors** — the same
representation the probe used. No GPU needed and no re-encoding: I just refit the linear head six
times against six different holdouts.

## Protocol

For each of the six generators g:

- **Train set:** all human articles (from `split == train`) + AI articles from the other five
  generators (from `split == train`).
- **Test set:** all human articles (from `split == test`) + AI articles from g (from `split ==
  test`).

Nothing from `val` participates. Pair-awareness is preserved by using the splits assigned in NB3, so
no fact card straddles the boundary. The classifier is the same L2 logistic regression as the NB5b
probe, so a drop from 99.5% to a much lower number here would mean the probe's headline was
propped up by having seen every generator during training.

## Setup

In [1]:
import pandas as pd, numpy as np, os, glob
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

SEED = 42
OUT_DIR = '/kaggle/working'

def find(preferred, *keywords):
    if os.path.exists(preferred):
        return preferred
    for kw in keywords:
        hits = [p for p in glob.glob('/kaggle/input/**/*', recursive=True) if kw in p]
        if hits:
            print(f'(resolved {kw} -> {hits[0]})'); return hits[0]
    raise FileNotFoundError(preferred)

DATA  = find('/kaggle/input/notebooks/bahaaqassem/nb3-build-dataset/dataset.parquet', 'dataset')
# NB5b saves this file inside the working directory of its own run; if I'm re-running here I
# need to upload it as a Kaggle dataset first (e.g. `aigt-arabert-cls`).
EMB_P = find('/kaggle/input/notebooks/bahaaqassem/nb5b-arabert/arabert_cls_frozen.npy', 'arabert_cls_frozen', 'arabert_cls')

df = pd.read_parquet(DATA)
emb = np.load(EMB_P)
assert len(df) == len(emb), f'row/embedding mismatch: {len(df)} vs {len(emb)}'
print(f'dataset {df.shape} | embeddings {emb.shape}')

gens = sorted(df.loc[df['label']==1, 'generator'].unique().tolist())
print('generators:', gens)

dataset (7101, 7) | embeddings (7101, 768)
generators: ['deepseek', 'gemini', 'gpt', 'opus', 'qwen', 'sonnet']


## The LOGO loop

Six folds, one per generator. For each fold I fit logistic regression on the split-train + non-held
generators, score on the split-test human articles plus the held generator's split-test articles,
and record the full metric suite plus the per-class recalls (so I can see if human or AI is the one
being missed).

In [2]:
def rows_train(g):
    m = (df['split'] == 'train') & ((df['label']==0) | (df['generator'] != g))
    return np.where(m)[0]

def rows_test(g):
    m = (df['split'] == 'test') & ((df['label']==0) | (df['generator'] == g))
    return np.where(m)[0]

results = []
for g in gens:
    tr_idx = rows_train(g); te_idx = rows_test(g)
    Xtr, ytr = emb[tr_idx], df['label'].iloc[tr_idx].to_numpy()
    Xte, yte = emb[te_idx], df['label'].iloc[te_idx].to_numpy()
    clf = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED)
    clf.fit(Xtr, ytr)
    pred  = clf.predict(Xte)
    proba = clf.predict_proba(Xte)[:, 1]
    cm = confusion_matrix(yte, pred, labels=[0, 1])
    results.append({
        'held_out':      g,
        'n_train':       int(len(tr_idx)),
        'n_test_human':  int((yte == 0).sum()),
        'n_test_ai':     int((yte == 1).sum()),
        'accuracy':      accuracy_score(yte, pred),
        'precision':     precision_score(yte, pred),
        'recall':        recall_score(yte, pred),
        'macro_f1':      f1_score(yte, pred, average='macro'),
        'auc_roc':       roc_auc_score(yte, proba),
        'human_recall':  cm[0,0] / max(cm[0].sum(), 1),
        'ai_recall':     cm[1,1] / max(cm[1].sum(), 1),
    })

res = pd.DataFrame(results)

## Results and interpretation

In [3]:
show = res.copy()
for c in ['accuracy','precision','recall','macro_f1','auc_roc','human_recall','ai_recall']:
    show[c] = (100 * show[c]).round(1)

print(show[['held_out','n_test_ai','ai_recall','human_recall','macro_f1','auc_roc']].to_string(index=False))
print()
print(f"mean macro-F1 across folds: {100*res['macro_f1'].mean():.1f}%")
print(f"std  macro-F1 across folds: {100*res['macro_f1'].std():.1f}%")
print(f"min  macro-F1 (worst fold): {100*res['macro_f1'].min():.1f}% ({res.loc[res['macro_f1'].idxmin(),'held_out']} held out)")
print()
print('reference points:')
print("  NB5b frozen probe (all generators seen in train): 99.5% macro-F1")
print("  NB5a statistical-only (best):                     84.7% macro-F1")
print("  NB3 cheap-signal control:                         68.8% macro-F1")
print("  TF-IDF LOGO proxy (earlier diagnostic):           99.0% mean macro-F1")

drop = 100 * (0.995 - res['macro_f1'].mean())
print(f"\ngap vs the seen-all probe: {drop:.1f} points")
if drop < 3:
    verdict = 'AraBERT frozen [CLS] generalizes cleanly across held-out generators.'
elif drop < 10:
    verdict = 'AraBERT frozen [CLS] generalizes but with a mild per-generator dependence.'
else:
    verdict = 'AraBERT frozen [CLS] leans substantially on per-generator fingerprints.'
print(f'verdict: {verdict}')

res.to_parquet(f'{OUT_DIR}/nb5b_arabert_logo.parquet', index=False)
print(f"\nsaved {OUT_DIR}/nb5b_arabert_logo.parquet")

held_out  n_test_ai  ai_recall  human_recall  macro_f1  auc_roc
deepseek        135       94.1          99.6      97.6     99.8
  gemini         63      100.0          99.4      98.7    100.0
     gpt         68       79.4          99.6      92.8     99.5
    opus         63      100.0          99.6      99.1    100.0
    qwen         88       95.5          99.6      98.0     99.8
  sonnet        137       99.3          99.6      99.3    100.0

mean macro-F1 across folds: 97.6%
std  macro-F1 across folds: 2.4%
min  macro-F1 (worst fold): 92.8% (gpt held out)

reference points:
  NB5b frozen probe (all generators seen in train): 99.5% macro-F1
  NB5a statistical-only (best):                     84.7% macro-F1
  NB3 cheap-signal control:                         68.8% macro-F1
  TF-IDF LOGO proxy (earlier diagnostic):           99.0% mean macro-F1

gap vs the seen-all probe: 1.9 points
verdict: AraBERT frozen [CLS] generalizes cleanly across held-out generators.

saved /kaggle/working/nb5

## Notes

- **What the number means.** The gap between this LOGO mean and the 99.5% frozen probe is the
  quantitative form of the earlier concern: how much of the probe's headline came from seeing every
  generator in training. A small gap (say, under three points) means AraBERT's frozen [CLS] is
  learning a genuinely generator-agnostic human/AI signal; a large gap means it is doing something
  closer to per-generator memorization.

- **Human recall vs AI recall per fold.** Where the model fails matters. If AI recall drops sharply
  only when a specific generator is held out (say, Opus, whose stylistic fingerprint was closest to
  human in NB5a), that is the interesting result for the discussion. Human recall should stay
  roughly constant across folds because the human class is identical in every fold; a wobble there
  would flag something else.

- **This is not the fine-tuned number.** Full fine-tuning may or may not tighten this gap; that
  belongs to the second track in NB5b once it is run in earnest.

- **For the thesis.** Report the frozen-probe macro-F1, the mean LOGO macro-F1, the standard
  deviation and the worst-fold value, in that order. The pair `(99.5%, mean, worst)` tells a clearer
  robustness story than any single headline number.